In [1]:
import pandas as pd
import matplotlib as plt
from pyvis import network as net
import networkx as nx

# toon alle columns
pd.set_option('display.max_columns', None)

In [ ]:
ai = pd.read_excel("inputs/TrajectenAI2019-21.xlsx")
ai.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694 entries, 0 to 693
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student                     694 non-null    object 
 1   Opleiding ID                693 non-null    float64
 2   Opleiding Omschrijving      693 non-null    object 
 3   Startjaar(4)                693 non-null    float64
 4   Startjaar                   693 non-null    object 
 5   Eindjaar                    693 non-null    object 
 6   Doorloop: Diploma behaald?  693 non-null    object 
 7   Aantal studenten            694 non-null    int64  
 8   Doorloop (KF)               694 non-null    float64
dtypes: float64(3), int64(1), object(5)
memory usage: 48.9+ KB


In [3]:
met_traject = ai[ai.duplicated(subset=['Student'], keep=False)]
ai_uni_studs = ai['Student'].drop_duplicates()
met_tr_uni_studs = met_traject['Student'].drop_duplicates()
uni_opls = met_traject['Opleiding ID'].drop_duplicates()
print(f"Totaal unieke studenten: {len(ai_uni_studs)},\nunieke studenten met traject: {len(met_tr_uni_studs)},\nunieke opleidingen: {len(uni_opls)}")

Totaal unieke studenten: 344,
unieke studenten met traject: 165,
unieke opleidingen: 112


In [ ]:
# In de juiste volgorde
opl_gr = met_traject.groupby(['Opleiding ID', 'Opleiding Omschrijving']).count()
opl_d = opl_gr.to_dict()['Student']
nodes, values, labels, colors = [], [], [], []
for key in opl_d:
    nodes.append(int(key[0]))
    labels.append(key[1])
    values.append(opl_d[key])

nodes_str = [str(x) for x in nodes]

# kleurekes
for v in values:
    if v < 2:
        colors.append('#f19167')
    elif v < 5:
        colors.append('#51cb9e')
    elif v < 11:
        colors.append('#5f97d7')
    else:
        colors.append('#f3df5f')

gr=net.Network(notebook=True, cdn_resources='in_line')
gr.add_nodes(nodes, value=values, label=labels, title=nodes_str, color=colors)

# Adding edges
from collections import defaultdict

trajecten_by_student = met_traject[["Student", "Opleiding ID"]].groupby("Student")
t_b_s_dict = {student: opleiding["Opleiding ID"].to_list() for student, opleiding in trajecten_by_student}

def return_zero():
    return(0)
edges = defaultdict(return_zero)

for opleidingen in t_b_s_dict.values():
    if (nr_opleidingen := len(opleidingen)) > 1:
        for o_index in range(0, nr_opleidingen-1):
            edges[(opleidingen[o_index], opleidingen[o_index+1])] += 1
            
for edge in edges:
    gr.add_edge(edge[0], edge[1], value=edges[edge])

gr.show("outputs/MNM_AI1921.html")

MNM_AI1921.html
